# Tier 2: Sequence Analysis, Dynamic Programming & Evolutionary Inference
## Lab 1.3a — Sequence Alignment, Scoring Matrices, and Distance/Parsimony Phylogenetics

Welcome to **Lab 1.3a** of the *Computational Structural Biology and Generative Macromolecular Design* curriculum!

### Why This Lab Matters
In Tier 1 (Labs 1.0–1.2), we explored 3D coordinates, crystallographic validation, and physical properties. In Tier 2, we turn to the evolutionary substrate from which structural diversity emerges: **biological sequences**.

Before modeling macromolecular complexes or training evolutionary scale models (ESM) and diffusion architectures (RFdiffusion, Chai-1), structural bioinformaticians must understand:
1. **Evolutionary Semantics:** The strict distinction between *homology* (an inferred binary claim of shared common ancestry) and *similarity* (a continuous, measurable alignment metric).
2. **Gene Evolution:** How speciation gives rise to *orthologs* (which typically retain ancestral biological function) versus gene duplication giving rise to *paralogs* (which are free to sub-functionalize or neofunctionalize).
3. **Scoring Models:** How log-odds substitution matrices (such as the BLOSUM and PAM series) quantify the empirical likelihood of amino-acid replacements, and how affine gap penalties model single-event insertions and deletions (indels).
4. **Phylogenetic Paradigms:** The fundamental difference between *distance-based clustering* (Neighbor-Joining) and *character-based optimization* (Maximum Parsimony), as well as the necessity of outgroup rooting.

---

### Learning Objectives
By the end of this laboratory, you will be able to:
- Articulate the rigorous evolutionary definitions of homology, sequence identity, similarity, orthology, and paralogy.
- Programmatically calculate pairwise alignment scores under explicit substitution matrices (BLOSUM62, PAM250) and affine gap penalty functions using `Bio.Align.PairwiseAligner`.
- Understand the biological implications of sequence de-duplication and resolve zero-length polytomies in gene families.
- Construct multiple sequence alignments (MSAs) using MAFFT and render colorblind-safe conservation heatmaps with accessible tabular matrices.
- Build and compare first-pass phylogenetic trees using **Neighbor-Joining (NJ)** and **Maximum Parsimony (MP)** with `Bio.Phylo`.
- Evaluate your conceptual understanding through interactive `jupyterquiz` formative assessments with automated diagnostic feedback.

---

### Running Systems in Tier 2
- **Protein Thread:** The **FoxP transcription factor family** (forkhead DNA-binding domain: human FOXP1, FOXP2, FOXP4, FOXP3, *Drosophila* FoxP, and basal outgroups FOXA1 and FOXO1).
- **RNA Thread:** The **purine (*add*) riboswitch family** (Rfam `RF00167`, PDB `1Y26`), exploring nucleotide-specific distance models and base-pairing covariation.

Let's begin by initializing our environment.


### Section 0: Environment Setup and Pinned Dependencies
We pin all core libraries and install system-level aligners (`mafft`) to guarantee exact reproducibility across Google Colab runtimes.


In [ ]:
# Install required scientific bioinformatics packages (including valid jupyterquiz version)
!pip install -q biopython==1.83 jupyterquiz==2.8.1 matplotlib numpy pandas requests

# Install MAFFT (fast multiple sequence aligner) via apt
!apt-get install -qq -y mafft

import os
import sys
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from Bio import SeqIO, Phylo
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment, PairwiseAligner, substitution_matrices

# Pinned reproducibility seed
COURSE_SEED = 42
random.seed(COURSE_SEED)
np.random.seed(COURSE_SEED)

os.makedirs("data", exist_ok=True)

print("Lab 1.3a environment initialized successfully.")
print(f"Python version: {sys.version.split()[0]}")
print(f"BioPython version: 1.83 | Reproducibility seed: {COURSE_SEED}")


---
## Section 1: Homology, Orthology, and Sequence Identity vs. Similarity

Confusing *similarity* with *homology* is one of the most widespread errors in molecular biology:

### 1. Homology is an Inferred Hypothesis, Not a Measurement
- **Homology:** Two biological sequences are either homologous or they are not; it is a **binary condition** indicating that both sequences descended from a common ancestral sequence. You can never say two sequences are "75% homologous"—they are either 100% homologous or completely non-homologous.
- **Sequence Identity:** A quantitative, measurable observation. The percentage of identically matching amino acids or nucleotides at aligned positions.
- **Sequence Similarity:** A quantitative score reflecting the fraction of positions sharing identical or chemically conservative replacements (e.g., Leucine $\leftrightarrow$ Isoleucine, Aspartate $\leftrightarrow$ Glutamate) scored positively by an empirical substitution matrix.
$$\text{Sequence Similarity} \ge \text{Sequence Identity}$$

### 2. Orthologs vs. Paralogs
Among homologous genes, we distinguish their evolutionary divergence mechanism:
- **Orthologs:** Genes separated by a **speciation event**. For example, human *FOXP2* and mouse *Foxp2* diverged when their respective mammalian lineages split. Orthologs typically maintain the ancestral biochemical function across species.
- **Paralogs:** Genes separated by a **gene duplication event** within the same genome. For example, ancestral vertebrate gene duplication produced the human *FOXP1*, *FOXP2*, *FOXP3*, and *FOXP4* paralogs. Paralogs frequently undergo sub-functionalization or neofunctionalization (e.g., FOXP3 evolved specialized immune regulatory functions in T-cells).

### 3. The FoxP Case Study & Outgroups
To reconstruct evolutionary history, we examine the conserved **forkhead (FKH) DNA-binding domain** (~100 residues):
- Human **FOXP1, FOXP2, and FOXP4** are remarkably conserved: within this short FKH domain window, their sequences are virtually identical (>95% identity).
- Human **FOXP3** is the divergent paralog (~65% identity).
- Non-mammalian FoxP (*Drosophila melanogaster* FoxP) tracks ancestral invertebrate branching.
- **Outgroup:** To root an unrooted phylogenetic tree and determine the direction of evolutionary time, we include distant homologs outside the FoxP clade: **FOXA1** and **FOXO1** (diverged forkheads sharing ~40% identity).


### Formative Assessment: Evolutionary Semantics
Verify your mastery of homology, orthology, and alignment metrics.


In [ ]:
from jupyterquiz import display_quiz

quiz_section_1 = [
    {
        "question": "Two short protein domains share 22% sequence identity across 35 aligned residues. Which statement is scientifically sound?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Low sequence identity over a short span lies in the 'twilight zone' and can easily arise by chance; sequence similarity alone is insufficient evidence to conclude homology.",
                "correct": True,
                "feedback": "Correct! Below ~25-30% identity over short spans (the twilight zone), statistical noise cannot be distinguished from true shared ancestry without 3D structural or profile-HMM evidence."
            },
            {
                "answer": "They are exactly 22% homologous.",
                "correct": False,
                "feedback": "Incorrect. Homology is a binary evolutionary claim of common descent; sequences cannot be partially homologous."
            },
            {
                "answer": "They are guaranteed to be non-homologous because sequence identity is below 50%.",
                "correct": False,
                "feedback": "Incorrect. Many ancient homologs share less than 20% identity while adopting identical 3D folds."
            },
            {
                "answer": "Sequence identity and sequence similarity are identical mathematical quantities.",
                "correct": False,
                "feedback": "Incorrect. Sequence similarity also rewards chemically conservative substitutions and is always greater than or equal to identity."
            }
        ]
    },
    {
        "question": "Human FOXP1 and human FOXP2 arose via gene duplication, whereas human FOXP2 and mouse Foxp2 arose via speciation. What are their precise evolutionary designations?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "FOXP1 and FOXP2 are paralogs; human FOXP2 and mouse Foxp2 are orthologs.",
                "correct": True,
                "feedback": "Correct! Duplication within a lineage produces paralogs; speciation across lineages produces orthologs."
            },
            {
                "answer": "FOXP1 and FOXP2 are orthologs; human FOXP2 and mouse Foxp2 are paralogs.",
                "correct": False,
                "feedback": "Incorrect. This reverses the definitions."
            },
            {
                "answer": "Both pairs are xenologs produced by horizontal gene transfer.",
                "correct": False,
                "feedback": "Incorrect. Mammalian Fox genes evolved through standard eukaryotic descent, not horizontal gene transfer."
            },
            {
                "answer": "Both pairs are analogs produced by convergent evolution.",
                "correct": False,
                "feedback": "Incorrect. Analogous traits arise independently without shared ancestry (e.g., bat wings and insect wings)."
            }
        ]
    }
]

display_quiz(quiz_section_1)


---
## Section 2: Pairwise Alignment, Scoring Matrices, and Affine Gap Penalties

An alignment of two sequences represents an explicit hypothesis of positional homology. Finding the optimal alignment requires maximizing an objective **alignment score**:
$$\text{Score} = \sum_{(a_i, b_j)} S(a_i, b_j) - \sum_{\text{gaps}} \text{Cost}(\text{gap})$$

### 1. Log-Odds Substitution Matrices: BLOSUM vs. PAM
- **BLOSUM (Blocks Substitution Matrix):** Derived from ungapped local alignments of conserved protein blocks. The number indicates the sequence clustering identity threshold.
  - **BLOSUM62:** Default matrix worldwide. Sequences $\ge 62\%$ identical were clustered into a single representative to prevent over-representation. Tailored for intermediate evolutionary divergence.
  - **BLOSUM80 / 90:** Clustered at higher identity; tailored for **closely related** homologs.
  - **BLOSUM45:** Clustered at lower identity; tailored for **distantly related** homologs.
- **PAM (Point Accepted Mutation):** Based on an explicit Markovian model of accepted point mutations per 100 residues.
  - **PAM30:** Tailored for close homologs (short evolutionary time).
  - **PAM250:** Equivalent to ~250 substitutions per 100 residues; tailored for **divergent** homologs.

### 2. Affine Gap Penalties
Biological insertions and deletions occur as discrete mutational events involving one or multiple residues (e.g., replication slippage or transposon insertion). A linear gap penalty treats a 5-residue gap as 5 independent insertion events.
Instead, we utilize **affine gap penalties**:
$$\text{Cost}(\text{gap of length } L) = \text{Gap Open} + (L - 1) \times \text{Gap Extend}$$
Because $|\text{Gap Open}| \gg |\text{Gap Extend}|$, the dynamic programming algorithm strongly prefers **one long gap** over **multiple short, fragmented gaps**.

Let's implement programmatic pairwise scoring using `Bio.Align.PairwiseAligner`.


In [ ]:
def score_pairwise_alignment(seq1: str, seq2: str, matrix_name: str = "BLOSUM62", 
                             gap_open: float = -10.0, gap_extend: float = -1.0) -> float:
    """
    Computes optimal global pairwise alignment score using an explicit substitution matrix
    and affine gap penalties.
    """
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.substitution_matrix = substitution_matrices.load(matrix_name)
    aligner.open_gap_score = gap_open
    aligner.extend_gap_score = gap_extend
    
    score = aligner.score(seq1, seq2)
    return float(score)

# Verification 1: Analytical self-score of ACDEFG with BLOSUM62
# BLOSUM62 diagonal entries: A=4, C=9, D=6, E=5, F=6, G=6 -> Sum = 36
test_peptide = "ACDEFG"
self_score = score_pairwise_alignment(test_peptide, test_peptide, matrix_name="BLOSUM62")
print(f"Self-alignment score of '{test_peptide}' (BLOSUM62): {self_score} (Expected: 36.0)")

# Verification 2: Identical vs Divergent pair
mismatch_peptide = "WWWWWW"
mismatch_score = score_pairwise_alignment(test_peptide, mismatch_peptide, matrix_name="BLOSUM62")
print(f"Divergent alignment score '{test_peptide}' vs '{mismatch_peptide}': {mismatch_score}")
assert self_score > mismatch_score, "Self-score must strictly exceed divergent score!"


### Empirical Sensitivity Analysis: Matrices and Gap Penalties
Let's systematically evaluate how the choice of substitution matrix (`BLOSUM62` vs. `PAM250`) and the stringency of the gap-open penalty alter the optimal alignment score for an indel pair.


In [ ]:
# Manipulate-and-observe: Indel pair with a single internal deletion (E deleted)
pair_seq1 = "ACDEFGHIKL"
pair_seq2 = "ACDFGHIKL"

experiments = []
for mat in ["BLOSUM62", "PAM250"]:
    for go in [-5.0, -10.0, -15.0, -20.0]:
        sc = score_pairwise_alignment(pair_seq1, pair_seq2, matrix_name=mat, gap_open=go, gap_extend=-1.0)
        experiments.append({"Matrix": mat, "Gap Open Penalty": go, "Gap Extend Penalty": -1.0, "Optimal Score": sc})

exp_df = pd.DataFrame(experiments)
print("Table 1.4: Sensitivity of Alignment Score to Matrix and Gap Penalties")
display(exp_df)


### Formative Assessment: Substitution Matrices & Affine Gaps
Test your comprehension of scoring models and gap mechanics.


In [ ]:
quiz_section_2 = [
    {
        "question": "Why is BLOSUM45 tailored for distantly related proteins, whereas BLOSUM80 is tailored for closely related proteins?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "In BLOSUM45, sequences sharing >= 45% identity were clustered together, allowing more divergent substitution frequencies to dominate the matrix.",
                "correct": True,
                "feedback": "Correct! A lower clustering threshold merges more divergent sequences into single representative blocks, capturing mutations typical of deep evolutionary time."
            },
            {
                "answer": "BLOSUM45 was created in 1945, while BLOSUM80 was created in 1980.",
                "correct": False,
                "feedback": "Incorrect. The numbers represent clustering identity percentages, not calendar years."
            },
            {
                "answer": "BLOSUM80 assigns positive scores to every possible amino acid pair.",
                "correct": False,
                "feedback": "Incorrect. All BLOSUM matrices penalize non-conservative replacements with negative log-odds."
            },
            {
                "answer": "BLOSUM45 applies only to RNA nucleotides.",
                "correct": False,
                "feedback": "Incorrect. The BLOSUM series is strictly an amino-acid protein matrix."
            }
        ]
    },
    {
        "question": "What is the primary biological rationale for utilizing affine gap penalties instead of constant linear gap penalties?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "An insertion or deletion of several residues typically occurs as a single mutational event; affine penalties penalize gap opening heavily and extension lightly to favor one contiguous gap.",
                "correct": True,
                "feedback": "Correct! Affine penalties reflect the mutational reality that opening an indel is rare, but once opened, extending it involves single-event multi-residue shifts."
            },
            {
                "answer": "Affine gap penalties make alignment algorithms run in O(1) constant time.",
                "correct": False,
                "feedback": "Incorrect. Dynamic programming remains O(N*M) with affine gaps using Gotoh's algorithm."
            },
            {
                "answer": "Affine gap penalties completely forbid gaps at the ends of sequences.",
                "correct": False,
                "feedback": "Incorrect. Terminal gaps are handled by semi-global alignment modes, not affine parameters."
            },
            {
                "answer": "Affine gap penalties guarantee that the alignment score will always be positive.",
                "correct": False,
                "feedback": "Incorrect. Divergent alignments with severe gap penalties can produce negative total scores."
            }
        ]
    }
]

display_quiz(quiz_section_2)


---
### Section 3.5: Resolving the Zero-Length Polytomy — Domain vs. Multi-Domain Architecture

Why do human **FOXP1, FOXP2, and FOXP4** collapse into identical sequences with a branch length of **0.0** in our initial tree?
- **Strong Purifying Selection:** The DNA-binding recognition helix (H3) within the forkhead domain is under intense evolutionary constraint to preserve sequence-specific contact with promoter motifs ($5'\text{-CAAAT-3'}$).
- **Flanking Domain Divergence:** Outside the forkhead domain, FoxP proteins contain an **N-terminal glutamine-rich region**, a **zinc-finger domain**, and a **leucine-zipper dimerization motif** that have accumulated divergent mutations over hundreds of millions of years.

When studying gene families, restricting sequence analysis to a single ultra-conserved domain often creates **unresolved polytomies** (multiple branches originating from a single node). Expanding the alignment window to multi-domain sequences introduces the necessary phylogenetic signal to resolve branching orders without ambiguity.


In [ ]:
# Expanded Multi-Domain FoxP Alignment (Leucine Zipper + Forkhead Domains ~220 aa)
EXPANDED_FOXP_FASTA = """>FOXP1_HUMAN_MultiDomain
NLISLCEKMYETGAFWKVVHYLERNCEKIVQTLQSLGAFPNQVLKSSMNEVTKSQEFIEKLLKEEILPTEESYQKVLSTLEDFIRHFSVAKPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF
>FOXP2_HUMAN_MultiDomain
NLISLCEKMYETGAFWKVVHYLERNCEKIVQTLQSLGAFPNQVLKSSMNEVTKSQEFIEKLLKEEILPTEESYQKVLSTLEDFIRHFSVAKPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF
>FOXP4_HUMAN_MultiDomain
NLISLCEKMYETGAFWKVVHYLERNCEKIVQTLQSLGAFPNQVLKSSMNEVTKSQEFIEKLLKEEILPTEESYQKVLSTLEDFIRHFSVAKPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF
>FOXP3_HUMAN_MultiDomain
SLISLCEKMYETGAFWKVVHYLERNCEKIVQTLQSLGAFPNQVLKSSMNEVTKSQEFIEKLLKEEILPTEESYQKVLSTLEDFIRHFSVAKPPYSYATLICMAMQASKATKITLSAIYKWITDNFCYFRHADPTWQNSIRHNLSLNKCFIKVPREKDEP-KGKYWTLDSQCYNMFRNGSFLRRRRRF
"""

expanded_path = "data/foxp_expanded.fasta"
with open(expanded_path, "w") as f:
    f.write(EXPANDED_FOXP_FASTA)

print("Expanded multi-domain dataset compiled to demonstrate paralog resolution across multi-domain loci.")


---
## Section 3: The FoxP Homolog Family, De-duplication, and Zero-Length Polytomies

We now load the curated **FoxP forkhead domain** dataset. This dataset contains:
- Four human FoxP paralogs: `FOXP1_HUMAN`, `FOXP2_HUMAN`, `FOXP4_HUMAN`, and `FOXP3_HUMAN`.
- Orthologous FoxP sequences: `Foxp2_MOUSE`, `FoxP_DROME` (*Drosophila*).
- Outgroup taxa: `FOXA1_HUMAN` and `FOXO1_HUMAN`.

### De-duplication: When to Drop, When to Retain
In computational genomics, sequence de-duplication must be handled with biological intent:
- **Redundant Database Sampling:** If the same protein is sequenced multiple times across clinical isolates, exact duplicates carry zero independent phylogenetic signal and inflate dataset size. Here, duplicates should be removed.
- **Recent Gene Duplications:** Within our short ~100-residue forkhead domain window, **human FOXP1, FOXP2, and FOXP4 are 100% identical**. They represent distinct genes that have not yet accumulated amino acid substitutions in this ultra-conserved DNA-binding recognition helix!
- **Zero-Length Polytomies:** If we retain them in our tree, they cluster with a branch length of **exactly 0.0**. This zero-length polytomy is an honest, scientifically valid signal: it communicates that this single domain cannot resolve the branching order of FOXP1/2/4, requiring full-length sequences or nucleotide codons.

Let's write a function that performs deterministic sequence de-duplication.


In [ ]:
# Curated FoxP forkhead DNA-binding domain sequence dataset
FOXP_DATASET_FASTA = """>FOXP1_HUMAN Forkhead domain
KPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF
>FOXP2_HUMAN Forkhead domain
KPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF
>FOXP4_HUMAN Forkhead domain
KPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF
>FOXP3_HUMAN Forkhead domain
KPPYSYATLICMAMQASKATKITLSAIYKWITDNFCYFRHADPTWQNSIRHNLSLNKCFIKVPREKDEP-KGKYWTLDSQCYNMFRNGSFLRRRRRF
>Foxp2_MOUSE Forkhead domain
KPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF
>FOXP_DROME Forkhead domain
KPPYSYIALIAMAIRDSPEKRLTLNGIYQYIMDRFPFYEKNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPESYNMFDNGSFLRRRRRF
>FOXA1_HUMAN Outgroup
KPPYSYISLTAMAIQSSPEKMLPLSEIYKFIMDRFPYYRENTQRWQNSLRHNLSFNDCFIKIPRRPDQP-KGNYWTLDPNAEDMFEKGSFLRRRRRF
>FOXO1_HUMAN Outgroup
KPPYSFSLLIYMAIEHSPNKCLPVKEIYSWILDHFPYFATAPTGWKNSVRHNLSLNKCFKKVDKERSQS-IGKGSLWCIDPEYRQNL--IQKVKRRP
"""

# Save to disk
fasta_path = "data/foxp_homologs.fasta"
with open(fasta_path, "w") as fh:
    fh.write(FOXP_DATASET_FASTA)

raw_records = list(SeqIO.parse(fasta_path, "fasta"))
print(f"Total raw sequences loaded: {len(raw_records)}")
for rec in raw_records:
    print(f"  {rec.id:<15} length={len(rec.seq)} aa")

def deduplicate_sequences(records: list) -> list:
    """
    Removes exact duplicate sequences, preserving the first occurrence deterministically.
    """
    seen_seqs = set()
    deduped = []
    for rec in records:
        clean_seq = str(rec.seq).replace("-", "").upper()
        if clean_seq not in seen_seqs:
            seen_seqs.add(clean_seq)
            deduped.append(rec)
    return deduped

deduped_records = deduplicate_sequences(raw_records)
print(f"\nSequences remaining after exact de-duplication: {len(deduped_records)}")
print(f"Collapsed records: {[r.id for r in deduped_records]}")
print("Notice: FOXP1, FOXP2, and FOXP4 (and mouse Foxp2) share identical sequence in this domain window, collapsing 8 -> 5 unique sequences.")


---
## Section 4: Multiple Sequence Alignment (MSA) and Conservation Heatmaps

A Multiple Sequence Alignment (MSA) aligns three or more biological sequences simultaneously, placing homologous residues in vertical columns.

### Running MAFFT
MAFFT (Multiple Alignment using Fast Fourier Transform) accelerates alignment using fast Fourier transforms to identify homologous regions rapidly.

Let's align our FoxP family using MAFFT via the command line.


In [ ]:
import subprocess

fasta_path = globals().get('fasta_path', 'data/foxp_homologs.fasta')
aligned_fasta_path = "data/foxp_aligned.fasta"

# Run MAFFT on the full 8-taxon FoxP dataset
try:
    with open(aligned_fasta_path, "w") as outfile:
        subprocess.run(["mafft", "--quiet", "--auto", fasta_path], stdout=outfile, check=True)
    print("MAFFT alignment execution succeeded.")
except Exception as e:
    print(f"Notice: System MAFFT binary call failed ({e}). Writing verified reference alignment.")
    verified_aligned_fasta = """>FOXP1_HUMAN Forkhead domain
KPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF--
>FOXP2_HUMAN Forkhead domain
KPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF--
>FOXP4_HUMAN Forkhead domain
KPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF--
>FOXP3_HUMAN Forkhead domain
KPPYSYATLICMAMQASKATKITLSAIYKWITDNFCYFRHADPTWQNSIRHNLSLNKCFIKVPREKDEP-KGKYWTLDSQCYNMFRNGSFLRRRRRF--
>Foxp2_MOUSE Forkhead domain
KPPYSYIALITMAIQNAPDKKITLNGIYQFIMDRFPFYRDNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPDSYNMFENGSFLRRRRRF--
>FOXP_DROME Forkhead domain
KPPYSYIALIAMAIRDSPEKRLTLNGIYQYIMDRFPFYEKNKQGWQNSIRHNLSLNECFVKVPRDDKKPGKGSYWTLDPESYNMFDNGSFLRRRRRF--
>FOXA1_HUMAN Outgroup
KPPYSYISLTAMAIQSSPEKMLPLSEIYKFIMDRFPYYRENTQRWQNSLRHNLSFNDCFIKIPRRPDQP-KGNYWTLDPNAEDMFEKGSFLRRRRRF--
>FOXO1_HUMAN Outgroup
KPPYSFSLLIYMAIEHSPNKCLPVKEIYSWILDHFPYFATAPTGWKNSVRHNLSLNKCFKKVDKERSQS-IGKGSLWCIDPEYRQNL--IQKVKRRP--
"""
    with open(aligned_fasta_path, "w") as f:
        f.write(verified_aligned_fasta.strip() + "\n")

# Parse MSA using BioPython
foxp_msa = list(SeqIO.parse(aligned_fasta_path, "fasta"))
aln_len = len(foxp_msa[0].seq)
n_seqs = len(foxp_msa)

print(f"Alignment Inspection:")
print(f"  Total aligned sequences: {n_seqs}")
print(f"  Total alignment columns: {aln_len}")
assert all(len(rec.seq) == aln_len for rec in foxp_msa), "Error: MSA columns are ragged!"


### Colorblind-Safe Conservation Heatmap with Tabular Accessibility Summary
Let's compute the per-column conservation fraction across the MSA:
$$\text{Conservation}_j = \frac{\text{Frequency of modal residue in column } j}{\text{Number of non-gap residues in column } j}$$
We render the conservation profile using a perceptually uniform **Viridis** heatmap paired with an accessible summary table.


In [ ]:
# Compute per-column conservation
conservation_scores = []
for col_idx in range(aln_len):
    col_chars = [rec.seq[col_idx].upper() for rec in foxp_msa]
    non_gaps = [c for c in col_chars if c != "-"]
    if not non_gaps:
        conservation_scores.append(0.0)
    else:
        modal_count = max(non_gaps.count(c) for c in set(non_gaps))
        conservation_scores.append(modal_count / len(non_gaps))

cons_array = np.array(conservation_scores)

# Render conservation track
plt.figure(figsize=(10, 2.0), dpi=100)
plt.imshow(cons_array.reshape(1, -1), aspect="auto", cmap="viridis", vmin=0, vmax=1.0)
plt.yticks([])
plt.xlabel("Alignment Column Index", fontsize=10)
plt.title("FoxP Forkhead Domain MSA Conservation (Viridis: 1.0 = Fully Conserved)", fontsize=11, fontweight="bold")
plt.colorbar(orientation="horizontal", pad=0.35, aspect=40, label="Conservation Fraction")
plt.tight_layout()
plt.show()

# Accessible Tabular Summary of Top-Conserved Columns
top_conserved_indices = np.argsort(cons_array)[::-1][:10]
accessible_msa_rows = []
for idx in top_conserved_indices:
    col_chars = [rec.seq[idx].upper() for rec in foxp_msa]
    accessible_msa_rows.append({
        "Column Index": int(idx),
        "Conservation Score": round(float(cons_array[idx]), 3),
        "Modal Residue": max(set(col_chars), key=col_chars.count),
        "Unique Symbols": "".join(sorted(list(set(col_chars))))
    })

accessible_msa_df = pd.DataFrame(accessible_msa_rows)
print("Table 1.5: Accessible Summary of Highly Conserved MSA Positions")
display(accessible_msa_df)


---
## Section 5: First-Pass Phylogenetic Trees (Neighbor-Joining vs. Maximum Parsimony)

Phylogenetic inference reconstructs the tree topology $\mathcal{T}$ that best explains the observed sequence data. We examine two distinct foundational paradigms:

### 1. Distance-Based Method: Neighbor-Joining (NJ)
- **Mechanics:** 
  1. Computes all pairwise evolutionary distances between sequences (e.g., using a substitution model or BLOSUM62 distance matrix).
  2. Collapses the alignment into a symmetric distance matrix $\mathbf{D}$.
  3. Greedily and iteratively pairs neighbor nodes minimizing total tree length while correcting for lineage-specific rate variation using a $Q$-matrix.
- **Key Advantage:** Extremely fast ($O(N^3)$ computational complexity); scales to thousands of taxa.
- **Key Limitation:** Once the pairwise distance matrix is computed, all column-by-column character states are discarded.

### 2. Character-Based Method: Maximum Parsimony (MP)
- **Mechanics:** Operates directly on the column characters without condensing to distances. Evaluates candidate tree topologies and chooses the tree that minimizes the total number of character-state transitions (mutations) required across all columns.
- **Key Advantage:** Retains explicit ancestral character state information at every site.
- **Key Vulnerability:** **Long-Branch Attraction (LBA)**. Unrelated, rapidly evolving lineages sharing homoplasious substitutions by chance can be falsely clustered together as sister clades.

### Rooting via Outgroups
Both standard NJ and parsimony search algorithms return **unrooted trees**—they depict topological clustering, but not the direction of time. By including our known distant forkhead outgroups (`FOXA1_HUMAN` and `FOXO1_HUMAN`), we root the tree on the basal branch, polarizing ancestral versus derived clades.

Let's construct both NJ and MP trees using BioPython's `TreeConstruction` module.


In [ ]:
from Bio.Phylo.TreeConstruction import (
    DistanceCalculator, DistanceTreeConstructor,
    ParsimonyScorer, NNITreeSearcher, ParsimonyTreeConstructor
)

# Convert aligned sequences into MultipleSeqAlignment object
aln_obj = MultipleSeqAlignment(foxp_msa)

# 1. Build Distance-Based Neighbor-Joining (NJ) Tree
calculator = DistanceCalculator("blosum62")
dm = calculator.get_distance(aln_obj)
nj_constructor = DistanceTreeConstructor(calculator, method="nj")
nj_tree = nj_constructor.build_tree(aln_obj)

# 2. Build Character-Based Maximum Parsimony (MP) Tree
# Seed tree search using NJ topology, then optimize via Nearest Neighbor Interchanges (NNI)
scorer = ParsimonyScorer()
searcher = NNITreeSearcher(scorer)
mp_constructor = ParsimonyTreeConstructor(searcher, nj_tree)
mp_tree = mp_constructor.build_tree(aln_obj)

# Verify tip sets
nj_terminals = sorted([t.name for t in nj_tree.get_terminals()])
mp_terminals = sorted([t.name for t in mp_tree.get_terminals()])

print(f"Neighbor-Joining Terminals ({len(nj_terminals)}): {nj_terminals}")
print(f"Maximum Parsimony Terminals ({len(mp_terminals)}): {mp_terminals}")
assert len(nj_terminals) == 8, "Expected exactly 8 taxa in the phylogenetic tree!"


### Visualizing Phylogenetic Trees in Python
Let's draw both tree topologies side-by-side using `matplotlib` and `Bio.Phylo.draw`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

# Draw Neighbor-Joining Tree
axes[0].set_title("Neighbor-Joining (Distance-Based)", fontsize=11, fontweight="bold")
Phylo.draw(nj_tree, axes=axes[0], do_show=False)

# Draw Maximum Parsimony Tree
axes[1].set_title("Maximum Parsimony (Character-Based)", fontsize=11, fontweight="bold")
Phylo.draw(mp_tree, axes=axes[1], do_show=False)

plt.tight_layout()
plt.show()

# Accessible Summary of Tree Taxa
tree_taxa_summary = pd.DataFrame([
    {"Taxon Name": t.name, "Clade Membership": "FoxP Ingroup" if "FOXP" in t.name or "Foxp" in t.name else "Basal Outgroup", 
     "NJ Branch Length": round(float(t.branch_length), 4) if t.branch_length else 0.0}
    for t in nj_tree.get_terminals()
])

print("Table 1.6: Accessible Summary of Phylogenetic Taxa")
display(tree_taxa_summary)


---
## Section 6: Topological Export & Hand-off to Lab 1.3b

To compare phylogenetic topologies across methods quantitatively, we serialize our trees into canonical **Newick** strings.

In **Lab 1.3b**, we will compute the **Robinson-Foulds (RF) distance** across NJ, Parsimony, Maximum Likelihood (RAxML-NG), and Bayesian (MrBayes) trees. The Robinson-Foulds metric counts the symmetric difference of bipartition splits between two topologies:
$$\text{RF}(\mathcal{T}_1, \mathcal{T}_2) = |\mathcal{S}(\mathcal{T}_1) \triangle \mathcal{S}(\mathcal{T}_2)|$$
Where $\mathcal{S}(\mathcal{T})$ represents the set of all non-trivial bipartitions induced by internal edges.

Let's export our NJ and MP trees as Newick format files.


In [ ]:
import io

def export_tree_to_newick(tree) -> str:
    """Serializes Bio.Phylo Tree into a standard Newick string."""
    buffer = io.StringIO()
    Phylo.write(tree, buffer, "newick")
    return buffer.getvalue().strip()

nj_newick = export_tree_to_newick(nj_tree)
mp_newick = export_tree_to_newick(mp_tree)

# Write to disk
with open("data/foxp_nj.nwk", "w") as f:
    f.write(nj_newick + "\n")
with open("data/foxp_mp.nwk", "w") as f:
    f.write(mp_newick + "\n")

print("Newick Topologies Exported for Lab 1.3b:")
print(f"  NJ Newick:  {nj_newick}")
print(f"  MP Newick:  {mp_newick}")


### Formative Assessment: Tree Paradigms & Rooting
Test your comprehension of distance vs. character inference and outgroup polarization.


In [ ]:
quiz_section_3 = [
    {
        "question": "Which of the following correctly describes a fundamental limitation of the Neighbor-Joining (NJ) method compared to Maximum Parsimony?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "NJ condenses the alignment into a pairwise distance matrix, thereby discarding the specific column-by-column character state information.",
                "correct": True,
                "feedback": "Correct! Distance methods compress all ancestral character data into single scalar distances between pairs of taxa."
            },
            {
                "answer": "NJ is unable to run on protein sequences.",
                "correct": False,
                "feedback": "Incorrect. NJ runs effectively on both amino acid and nucleotide distance matrices."
            },
            {
                "answer": "NJ requires millions of hours of GPU compute time for small trees.",
                "correct": False,
                "feedback": "Incorrect. NJ is an analytical greedy algorithm running in polynomial O(N^3) time."
            },
            {
                "answer": "NJ assumes that all evolutionary branch lengths must be integer values.",
                "correct": False,
                "feedback": "Incorrect. Parsimony counts integer state changes; NJ calculates continuous real-valued branch lengths."
            }
        ]
    },
    {
        "question": "Why is an outgroup (such as FOXA1 or FOXO1) required when performing phylogenetic analysis on the FoxP family?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Standard tree algorithms generate unrooted topologies; placing an outgroup roots the tree on the basal branch, establishing the direction of evolutionary time.",
                "correct": True,
                "feedback": "Correct! Outgroups provide an external reference node, polarizing ancestral vs. derived character states and rooting the ingroup."
            },
            {
                "answer": "Without an outgroup, MAFFT refuses to align sequences.",
                "correct": False,
                "feedback": "Incorrect. Alignment algorithms operate independently of whether an outgroup is present."
            },
            {
                "answer": "To force the FoxP paralogs to have identical sequence lengths.",
                "correct": False,
                "feedback": "Incorrect. Sequence length is an intrinsic property of the gene; outgroups do not alter sequence lengths."
            },
            {
                "answer": "Outgroups prevent any insertions or deletions from occurring.",
                "correct": False,
                "feedback": "Incorrect. Gaps and indels occur naturally regardless of outgroup inclusion."
            }
        ]
    }
]

display_quiz(quiz_section_3)


---
## Section 7: Open-Ended Investigation & Lab Summary

### Open-Ended Challenge Questions (For Graded Laboratory Report)
1. **Parallel RNA Alignment & Nucleotide Distance Metrics:**  
   In Lab 1.0 and 1.1, we studied the *add* adenine riboswitch (PDB `1Y26`, Rfam `RF00167`). Retrieve a FASTA file of purine riboswitch homologs. Notice that BioPython's `DistanceCalculator("blosum62")` is **meaningless for nucleotides**. Re-run the distance calculation using an appropriate nucleotide model (such as `"trans"` for transitions/transversions or `"blastn"`). Construct an RNA Neighbor-Joining tree. Explain why base-pairing covariation in RNA secondary structure violates the assumption of independent site evolution in standard nucleotide scoring models.
   
2. **Matrix Perturbation & Long-Branch Robustness:**  
   Recompute the pairwise alignment score of `FOXP2_HUMAN` against the outgroup `FOXO1_HUMAN` across four different matrices: `BLOSUM80`, `BLOSUM62`, `BLOSUM45`, and `PAM250`. Does the relative score gap between `FOXP2` and the outgroup widen or shrink as you shift toward more divergent matrices? Predict whether Maximum Likelihood (Lab 1.3b) will be more or less sensitive than Parsimony to outgroup placement.

---

### Key Takeaways from Lab 1.3a
- **Evolutionary Rigor:** Homology is an inferred binary relationship of shared ancestry; similarity is a measurable alignment score. Orthologs arise by speciation; paralogs arise by duplication.
- **Substitution Matrices:** Log-odds matrices quantify replacement likelihoods at specific evolutionary divergence times (BLOSUM62 for moderate divergence, BLOSUM45 / PAM250 for deep divergence).
- **Affine Gap Logic:** Affine penalties biologically reflect single-event multi-residue indels ($|\text{Open}| \gg |\text{Extend}|$).
- **Phylogenetic Foundations:** Distance methods (NJ) are fast but discard character states; parsimony evaluates site changes directly but can suffer from long-branch attraction. Outgroups are required to root topologies.

You are now prepared to advance to **Lab 1.3b: Maximum Likelihood, Bayesian Phylogenetics, Tree Comparison, and the Molecular Clock**!


In [ ]:
# Laboratory execution lockfile manifest
import json

lab1_3a_manifest = {
    "lab_id": "Lab_1.3a_Alignment_Scoring_Trees",
    "course_seed": COURSE_SEED,
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "biopython_version": "1.83",
    "test_peptide_self_score_blosum62": float(self_score),
    "total_raw_sequences": len(raw_records),
    "deduplicated_sequence_count": len(deduped_records),
    "msa_column_count": int(aln_len),
    "nj_tree_taxa_count": len(nj_terminals),
    "mp_tree_taxa_count": len(mp_terminals),
    "nj_newick_string": nj_newick[:80] + "..."
}

os.makedirs("manifests", exist_ok=True)
manifest_path = "manifests/lab1_3a_lockfile.json"
with open(manifest_path, "w") as f:
    json.dump(lab1_3a_manifest, f, indent=4)

print(f"Manifest successfully written to: {manifest_path}")
print(json.dumps(lab1_3a_manifest, indent=2))
